
This notebook builds **useful Stage 1 training triplets** from the **Tell Me Again** dataset only.

- **positive** = another summary of the **same Wikidata story**, but not a trivial near-duplicate
- **negative** = a summary from a **different story** that is **semantically close** to the anchor


## What the notebook does
- uses only **Tell Me Again**
- excludes **SemEval Wikidata IDs** to avoid leakage
- removes near-duplicate same-story summaries
- keeps only **non-trivial positive pairs**
- mines **hard negatives** with sentence embeddings
- writes:
  - `tell_me_again_triplets.jsonl`
  - `combined_extra_triplets.jsonl`

## Output format
Each row is written as:
```json
{
  "anchor_text": "...",
  "text_a": "...",
  "text_b": "...",
  "text_a_is_closer": true,
  "source": "tma_hard",
  "wikidata_id": "Q..."
}
```

In [1]:
import json
import random
import re
from itertools import combinations
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel


In [ ]:
TMA_PATH = "/kaggle/input/datasets/anisoaraana/similarity/tell_me_again"

SEMEVAL_DEV    = "/kaggle/input/datasets/anisoaraana/similarity/dev_track_a_labels.jsonl"
SEMEVAL_TEST_A = "/kaggle/input/datasets/anisoaraana/similarity/test_track_a_labels.jsonl"
SEMEVAL_TEST_B = "/kaggle/input/datasets/anisoaraana/similarity/test_track_b_labels.jsonl"

OUT_DIR = "/kaggle/working/"
TMA_OUT = OUT_DIR + "tell_me_again_triplets.jsonl"
ALL_OUT = OUT_DIR + "combined_extra_triplets.jsonl"

# CONFIG
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Keep Tell-Me-Again auxiliary, not dominant
MAX_TMA_TRIPLETS = 1200

# Story-level filtering
MAX_SUMMARIES_PER_STORY = 4
MAX_PAIRS_PER_STORY = 2
TMA_MIN_CHARS = 120
TMA_MAX_CHARS = 1800

# Positive pair constraints
POS_MIN_JACCARD = 0.12
POS_MAX_JACCARD = 0.75
POS_MIN_LEN_RATIO = 0.60
POS_MAX_LEN_RATIO = 1.80

# Hard-negative mining
TMA_NEIGHBORS = 40
NEG_MIN_JACCARD = 0.05
NEG_MAX_JACCARD = 0.45
NEG_MIN_LEN_RATIO = 0.55
NEG_MAX_LEN_RATIO = 1.90

# Small encoder for negative mining
TMA_MINING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TMA_MINING_BATCH_SIZE = 32

print(f"TMA_PATH: {TMA_PATH}")
print(f"Output: {TMA_OUT}")
print(f"Max triplets: {MAX_TMA_TRIPLETS}")


In [ ]:
# TEXT UTILITIES
def normalize_ws(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "").strip())

def tokenize_for_overlap(text: str):
    return re.findall(r"[a-z]+", normalize_ws(text).lower())

def token_jaccard(a: str, b: str) -> float:
    sa, sb = set(tokenize_for_overlap(a)), set(tokenize_for_overlap(b))
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / max(1, len(sa | sb))

def length_ratio(a: str, b: str) -> float:
    la, lb = max(1, len(a)), max(1, len(b))
    return min(la, lb) / max(la, lb)

def is_valid_summary(text: str) -> bool:
    text = normalize_ws(text)
    return TMA_MIN_CHARS <= len(text) <= TMA_MAX_CHARS

def dedup_summaries(texts):
    kept = []
    for t in texts:
        t = normalize_ws(t)
        if not is_valid_summary(t):
            continue
        is_dup = False
        for prev in kept:
            # near-duplicate summaries are not useful positives
            if token_jaccard(t, prev) > 0.88:
                is_dup = True
                break
        if not is_dup:
            kept.append(t)
        if len(kept) >= MAX_SUMMARIES_PER_STORY:
            break
    return kept


In [ ]:
# DATA LEAKAGE FILTER
def load_semeval_wikidata_ids():
    ids = set()

    for path in [SEMEVAL_DEV, SEMEVAL_TEST_A]:
        p = Path(path)
        if not p.exists():
            continue
        with open(p, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                for key in ["story_details_anchor", "story_details_a", "story_details_b"]:
                    wid = obj.get(key, {}).get("wikidata_id", "")
                    if wid:
                        ids.add(wid)

    p = Path(SEMEVAL_TEST_B)
    if p.exists():
        with open(p, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                for key in ["story_details", "story_details_anchor", "story_details_a", "story_details_b"]:
                    wid = obj.get(key, {}).get("wikidata_id", "")
                    if wid:
                        ids.add(wid)

    print(f"SemEval Wikidata IDs to exclude: {len(ids):,}")
    return ids


In [ ]:
# TELL ME AGAIN LOADING
def load_tma_stories(tma_root):
    root = Path(tma_root)
    summaries_dir = root / "summaries"
    if not summaries_dir.exists():
        summaries_dir = root

    if not summaries_dir.exists():
        raise FileNotFoundError(
            f"Tell Me Again summaries folder not found at: {tma_root}"
        )

    json_files = list(summaries_dir.glob("**/*.json"))
    print(f"Found {len(json_files):,} TMA JSON files")

    stories = {}
    skipped = 0

    for jf in tqdm(json_files, desc="Loading Tell Me Again"):
        try:
            with open(jf, encoding="utf-8") as f:
                obj = json.load(f)
        except Exception:
            skipped += 1
            continue

        wid = obj.get("wikidata_id", "")
        if not wid:
            skipped += 1
            continue

        texts = []

        # preferred source: English translations supplied by the dataset
        for lang, entry in obj.get("en_translated_summaries", {}).items():
            text = ""
            if isinstance(entry, dict):
                text = (entry.get("text") or "").strip()
            elif isinstance(entry, str):
                text = entry.strip()
            if text:
                texts.append(text)

        # fallback: native English summary if available
        if not texts:
            en_text = obj.get("summaries", {}).get("en", "")
            if isinstance(en_text, str) and en_text.strip():
                texts.append(en_text.strip())

        clean = dedup_summaries(texts)
        if len(clean) >= 2:
            stories[wid] = clean

    print(f"Skipped files: {skipped:,}")
    print(f"Stories with >=2 usable English summaries: {len(stories):,}")
    return stories


In [ ]:
# EMBEDDING-BASED HARD NEGATIVE MINING
def mean_pool(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).float()
    summed = (token_embeddings * mask).sum(1)
    counts = mask.sum(1).clamp(min=1e-9)
    return F.normalize(summed / counts, p=2, dim=1)

def encode_texts(texts, model_name=TMA_MINING_MODEL, batch_size=TMA_MINING_BATCH_SIZE):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    all_vecs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Encoding representative stories"):
            batch = texts[i:i+batch_size]
            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            ).to(device)
            out = model(**enc)
            vec = mean_pool(out, enc["attention_mask"])
            all_vecs.append(vec.cpu().numpy())

    return np.vstack(all_vecs).astype(np.float32)

def select_positive_pairs(summaries, max_pairs=MAX_PAIRS_PER_STORY):
    candidates = []
    for a, b in combinations(summaries, 2):
        jac = token_jaccard(a, b)
        lr = length_ratio(a, b)

        # keep same-story pairs that are clearly related but not trivial copies
        if POS_MIN_JACCARD <= jac <= POS_MAX_JACCARD and POS_MIN_LEN_RATIO <= lr <= POS_MAX_LEN_RATIO:
            # prefer moderate overlap: neither too close nor too distant
            score = abs(jac - 0.35)
            candidates.append((score, a, b))

    candidates.sort(key=lambda x: x[0])
    return [(a, b) for _, a, b in candidates[:max_pairs]]

def choose_negative_summary(anchor_text, positive_text, candidate_story_summaries):
    best = None
    best_score = -1.0

    pos_jac = token_jaccard(anchor_text, positive_text)

    for cand in candidate_story_summaries:
        jac = token_jaccard(anchor_text, cand)
        lr = length_ratio(anchor_text, cand)

        # hard but still clearly different
        if not (NEG_MIN_JACCARD <= jac <= NEG_MAX_JACCARD):
            continue
        if not (NEG_MIN_LEN_RATIO <= lr <= NEG_MAX_LEN_RATIO):
            continue

        # negative should usually be less aligned than the positive
        if jac >= pos_jac:
            continue

        # maximize allowed lexical overlap to make the negative hard
        score = jac
        if score > best_score:
            best = cand
            best_score = score

    return best


In [ ]:
# TRIPLET BUILDING
def build_tell_me_again_triplets():
    print("\n" + "=" * 60)
    print("BUILDING REDESIGNED TELL ME AGAIN TRAINING TRIPLETS")
    print("=" * 60)

    semeval_ids = load_semeval_wikidata_ids()
    stories = load_tma_stories(TMA_PATH)

    # remove any story already used in SemEval
    stories = {wid: summaries for wid, summaries in stories.items() if wid not in semeval_ids}
    print(f"After SemEval exclusion: {len(stories):,} stories")

    if not stories:
        print("No usable Tell Me Again stories found after leakage filtering.")
        Path(TMA_OUT).write_text("", encoding="utf-8")
        Path(ALL_OUT).write_text("", encoding="utf-8")
        return 0

    # representative summary per story for nearest-neighbor mining
    story_ids = list(stories.keys())
    rep_texts = [max(stories[wid], key=len) for wid in story_ids]

    rep_embs = encode_texts(rep_texts)
    nn = NearestNeighbors(
        n_neighbors=min(TMA_NEIGHBORS + 1, len(story_ids)),
        metric="cosine"
    )
    nn.fit(rep_embs)
    distances, indices = nn.kneighbors(rep_embs)

    written = 0
    skipped_no_pos = 0
    skipped_no_neg = 0

    with open(TMA_OUT, "w", encoding="utf-8") as fout:
        row_order = list(range(len(story_ids)))
        random.shuffle(row_order)

        for row_idx in tqdm(row_order, desc="Writing TMA hard triplets"):
            if written >= MAX_TMA_TRIPLETS:
                break

            wid = story_ids[row_idx]
            summaries = stories[wid]
            pos_pairs = select_positive_pairs(summaries, max_pairs=MAX_PAIRS_PER_STORY)

            if not pos_pairs:
                skipped_no_pos += 1
                continue

            neighbor_rows = [j for j in indices[row_idx] if j != row_idx]

            for anchor, positive in pos_pairs:
                if written >= MAX_TMA_TRIPLETS:
                    break

                negative = None
                for nb_row in neighbor_rows:
                    nb_wid = story_ids[nb_row]
                    cand_story = stories[nb_wid]
                    negative = choose_negative_summary(anchor, positive, cand_story)
                    if negative is not None:
                        break

                if negative is None:
                    skipped_no_neg += 1
                    continue

                # randomize position once; do not duplicate with swap
                if random.random() < 0.5:
                    text_a, text_b, closer = positive, negative, True
                else:
                    text_a, text_b, closer = negative, positive, False

                fout.write(json.dumps({
                    "anchor_text": anchor,
                    "text_a": text_a,
                    "text_b": text_b,
                    "text_a_is_closer": closer,
                    "source": "tma_hard",
                    "wikidata_id": wid,
                }, ensure_ascii=False) + "\n")

                written += 1

    # compatibility file for downstream training scripts
    Path(ALL_OUT).write_text(Path(TMA_OUT).read_text(encoding="utf-8"), encoding="utf-8")

    print(f"\nTriplets written        : {written:,}")
    print(f"Skipped (no pos pairs)  : {skipped_no_pos:,}")
    print(f"Skipped (no hard neg)   : {skipped_no_neg:,}")
    print(f"Saved                   : {TMA_OUT}")
    print(f"Combined copy           : {ALL_OUT}")
    return written


In [ ]:
def main():
    
    print("TELL ME AGAIN HARD-NEGATIVE TRIPLET BUILDER")
    n = build_tell_me_again_triplets()

main()
